# Followchon

## Variables

- Quantitative
    - Continue
        - Température 
- Qualitative
    - Ordinal
        - Heure
    - Nominal
        - Class (Noisette ou Stitch)
        - Zone (Clapier, Cachette, Fontaine, ...)
- Autres
    - Date
        - Date du jour
        - Date de la capture
    - Coordonnées
        - Coordonnée du chon (en valeur normal [0:1] )

## Récupération des détections

In [1]:
from django.db import connection
from detections.models import Detection
from configuration.models import Zone, Family
from datetime import datetime
from plotly.graph_objects import FigureWidget
from IPython.display import display
from ipywidgets import HBox, VBox, fixed, interactive_output

import os
import csv
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
import plotly.graph_objects as go

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

In [2]:
detections_query =  'SELECT d.id, ' + \
                    'STRFTIME("%Y-%m-%d %H:%M:%S", c.date) AS datetime, ' + \
                    'STRFTIME("%H", c.date) AS hour, ' + \
                    'STRFTIME("%Y%m%d%H", c.date) AS datehour_key, ' +  \
                       'z.id AS zone_id, ' +  \
                       'z.name AS zone_name, ' +  \
                       'f.[index] AS class_index, ' +  \
                       'f.name AS class_name, ' +  \
                       'd.center_x AS center_x_norm, ' +  \
                       'd.center_y AS center_y_norm ' +  \
                  'FROM detections_detection d ' +  \
                       'LEFT JOIN ' +  \
                       'detections_capture c ON d.capture_id = c.id ' +  \
                       'LEFT JOIN ' +  \
                       'configuration_family f ON d.family_id = f.id ' +  \
                       'LEFT JOIN ' +  \
                       'configuration_zone z ON d.zone_id = z.id ' +  \
                 'WHERE c.status == "archived" AND  ' +  \
                       'c.source == "vision" AND  ' +  \
                       '(f.[index] == 1 OR  ' +  \
                        'f.[index] == 2)  ' +  \
                 'ORDER BY c.date ASC '
                        
detections = Detection.objects.raw(detections_query)

zones_query = 'SELECT z.id, z.id AS zone_id, z.name AS zone_name FROM configuration_zone z ORDER BY z.id ASC'
zones = Zone.objects.raw(zones_query)

classes_query = 'SELECT f.id, f.[index] AS class_index, f.name AS class_name FROM configuration_family f ORDER BY f.id ASC'
classes = Family.objects.raw(classes_query)

def save_rows(rows, query, path): 
    columns = list()
    
    with connection.cursor() as cursor:
        cursor.execute(query)
        columns = [col[0] for col in cursor.description]
    
    with open(path, 'w+', newline='') as file:
        writer = csv.writer(file)

        writer.writerow(columns)
        
        for obj in rows:
            row = list()
            
            for col in columns:
                row.append(obj.__dict__[col])
                
            writer.writerow(row)
    
        print(f"{len(rows)} lignes exportées")

save_rows(detections, detections_query, 'data/detections.csv')
save_rows(zones, zones_query, 'data/zones.csv')
save_rows(classes, classes_query, 'data/classes.csv')

53474 lignes exportées


RecursionError: maximum recursion depth exceeded in comparison